In [ ]:
# =========================================
# 1. UPLOAD FILES
# =========================================
import os
import pandas as pd

try:
    from google.colab import files
    print("📂 Upload your CSV files...")
    uploaded = files.upload()
except:
    print("✅ Running locally - no upload needed")

print("\n📁 Files in directory:", os.listdir())


# =========================================
# 2. AUTO-RENAME (HANDLES (9), (1), etc.)
# =========================================
files_list = os.listdir()

def rename_if_found(keyword, new_name):
    for f in files_list:
        if keyword in f.lower():
            os.rename(f, new_name)
            print(f"✅ Renamed {f} → {new_name}")
            return new_name
    return None

orders_file = rename_if_found("orders", "orders.csv")
churn_file = rename_if_found("churn", "churn_labels.csv")

# fallback if already correctly named
orders_file = orders_file if orders_file else "orders.csv"
churn_file = churn_file if churn_file else "churn_labels.csv"


# =========================================
# 3. VALIDATE FILES
# =========================================
if not os.path.exists(orders_file) or not os.path.exists(churn_file):
    raise FileNotFoundError(
        f"❌ Required files not found.\nAvailable files: {os.listdir()}"
    )


# =========================================
# 4. LOAD DATA
# =========================================
orders = pd.read_csv(orders_file)
churn = pd.read_csv(churn_file)

print("✅ Files loaded successfully!")


# =========================================
# 5. DATE FORMATTING
# =========================================
orders['order_date'] = pd.to_datetime(orders['order_date'])
churn['snapshot_date'] = pd.to_datetime(churn['snapshot_date'])


# =========================================
# 6. FILTER DATA (NO LEAKAGE)
# =========================================
orders = orders.merge(churn[['customer_id','snapshot_date']], 
                      on='customer_id', how='left')

orders = orders[orders['order_date'] <= orders['snapshot_date']]


# =========================================
# 7. RFM FEATURE ENGINEERING
# =========================================
rfm = orders.groupby('customer_id').agg({
    'order_id': 'count',
    'gross_amount': 'sum',
    'order_date': 'max'
}).reset_index()

rfm.columns = ['customer_id', 'frequency', 'monetary', 'last_order_date']

# Merge snapshot
rfm = rfm.merge(churn[['customer_id','snapshot_date']], on='customer_id', how='left')

# Recency
rfm['recency'] = (rfm['snapshot_date'] - rfm['last_order_date']).dt.days


# =========================================
# 8. HANDLE MISSING VALUES
# =========================================
rfm.fillna(0, inplace=True)


# =========================================
# 9. SCALING
# =========================================
from sklearn.preprocessing import StandardScaler

X = rfm[['recency', 'frequency', 'monetary']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# =========================================
# 10. KMEANS CLUSTERING
# =========================================
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['cluster'] = kmeans.fit_predict(X_scaled)


# =========================================
# 11. SEGMENT LOGIC
# =========================================
cluster_summary = rfm.groupby('cluster').agg({
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).reset_index()

# Best customers = low recency, high frequency, high monetary
cluster_summary = cluster_summary.sort_values(
    by=['recency','frequency','monetary'],
    ascending=[True, False, False]
).reset_index(drop=True)

segment_labels = ['High Value', 'Regular', 'At Risk', 'Lost']
cluster_summary['segment_name'] = segment_labels

# Merge segments
rfm = rfm.merge(cluster_summary[['cluster','segment_name']], 
                on='cluster', how='left')


# =========================================
# 12. FINAL OUTPUT
# =========================================
final_segments = rfm[[
    'customer_id',
    'recency',
    'frequency',
    'monetary',
    'segment_name'
]]

# Save file
final_segments.to_csv('segments.csv', index=False)

print("\n✅ segments.csv generated successfully!")
print(final_segments.head())